# Automobile Fuel Efficiency Analytics with AI
### IBM SkillsBuild Data Analytics with AI Academic Internship — BharatCares x AICTE

**Author:** Siddaruda Satyappa Mantur
**Project:** Exploratory Data Analysis and Fuel Efficiency (MPG) Prediction using Regression

---

## 1. Business Problem

Automobile manufacturers and buyers want to understand which vehicle
specifications most strongly affect fuel efficiency, so design decisions
and purchase decisions can be made with data-backed confidence.

This project analyzes technical specifications of 398 cars (1970-1982) to:

1. Explore how engine size, weight, horsepower and other specs relate to
   fuel efficiency (miles per gallon).
2. Build regression models (Linear Regression and Random Forest) to
   predict a car's MPG from its specifications.
3. Provide recommendations for manufacturers on which design factors
   most influence fuel efficiency.

## 2. Dataset

- **Source:** Public "Auto MPG" dataset (UCI Machine Learning Repository,
  widely used for regression practice)
- **Rows:** 398 cars
- **Columns:** mpg, cylinders, displacement, horsepower, weight,
  acceleration, model_year, origin, car_name


## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
%matplotlib inline


## 4. Load and Inspect Data

In [ ]:
df = pd.read_csv("auto-mpg.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# horsepower has some missing values (marked blank in the original UCI dataset)
df.isnull().sum()


## 5. Data Cleaning

Missing horsepower values are imputed with the column median, since they represent only a small fraction of rows.

In [ ]:
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())
df.isnull().sum()


In [ ]:
df.describe()


## 6. Exploratory Data Analysis

### 6.1 Distribution of MPG

In [ ]:
plt.figure()
sns.histplot(df["mpg"], bins=20, kde=True, color="steelblue")
plt.title("Distribution of Fuel Efficiency (MPG)")
plt.xlabel("Miles per Gallon")
plt.show()


### 6.2 MPG vs Key Specifications

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.scatterplot(data=df, x="weight", y="mpg", ax=axes[0], color="darkorange")
axes[0].set_title("Weight vs MPG")
sns.scatterplot(data=df, x="horsepower", y="mpg", ax=axes[1], color="indianred")
axes[1].set_title("Horsepower vs MPG")
sns.scatterplot(data=df, x="displacement", y="mpg", ax=axes[2], color="seagreen")
axes[2].set_title("Engine Displacement vs MPG")
plt.tight_layout()
plt.show()


### 6.3 MPG by Number of Cylinders

In [ ]:
plt.figure()
sns.boxplot(data=df, x="cylinders", y="mpg", hue="cylinders", palette="Set2", legend=False)
plt.title("MPG by Number of Cylinders")
plt.show()


### 6.4 MPG Trend Over Model Years

In [ ]:
yearly_avg = df.groupby("model_year")["mpg"].mean()
plt.figure()
yearly_avg.plot(kind="line", marker="o", color="purple")
plt.title("Average MPG by Model Year (1970-1982)")
plt.xlabel("Model Year")
plt.ylabel("Average MPG")
plt.show()


### 6.5 Correlation Heatmap

In [ ]:
plt.figure()
corr = df[["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Between Specifications and MPG")
plt.show()


## 7. Key Insights from EDA

- **Weight, displacement, horsepower and cylinder count are all strongly
  negatively correlated** with MPG — heavier cars with bigger engines
  are less fuel-efficient, as expected.
- **Model year is positively correlated** with MPG — average fuel
  efficiency improved steadily from 1970 to 1982, likely reflecting the
  industry's response to the 1970s oil crisis and tightening emissions
  standards.
- Weight shows the strongest single correlation with MPG among all
  numeric features.


## 8. Predictive Modeling — MPG Prediction

We train two regression models using the numeric specification columns to predict MPG.

In [ ]:
features = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year", "origin"]
X = df[features]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape, " Test size:", X_test.shape)


### 8.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Linear Regression Results")
print("R2 Score:", round(r2_score(y_test, y_pred_lr), 3))
print("MAE:", round(mean_absolute_error(y_test, y_pred_lr), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_lr)), 2))


### 8.2 Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results")
print("R2 Score:", round(r2_score(y_test, y_pred_rf), 3))
print("MAE:", round(mean_absolute_error(y_test, y_pred_rf), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_rf)), 2))


### 8.3 Actual vs Predicted MPG (Random Forest)

In [ ]:
plt.figure()
plt.scatter(y_test, y_pred_rf, color="teal")
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--")
plt.xlabel("Actual MPG")
plt.ylabel("Predicted MPG")
plt.title("Actual vs Predicted MPG (Random Forest)")
plt.show()


### 8.4 Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values()
plt.figure()
importances.plot(kind="barh", color="darkorange")
plt.title("Feature Importance for Predicting MPG")
plt.xlabel("Importance")
plt.show()


## 9. Recommendations

1. **Prioritize weight reduction** — weight is the single strongest
   predictor of poor fuel efficiency; lighter materials and compact
   design yield the biggest MPG gains.
2. **Optimize engine displacement and cylinder count** — smaller,
   more efficient engines consistently correlate with higher MPG.
3. **Track year-over-year efficiency trends** — the steady MPG
   improvement from 1970-1982 shows that engineering and regulatory
   changes compound over time; continuous incremental improvement
   matters.
4. **Use the trained model for early-stage design estimates** — the
   Random Forest model can give designers a quick MPG estimate for a
   proposed set of specifications before physical prototyping.

## 10. Conclusion

This project demonstrates a complete data-analytics-with-AI workflow on
automotive specification data: data cleaning (handling missing
horsepower values), exploratory analysis, visualization, and two
regression models for predicting fuel efficiency. The Random Forest
model outperformed Linear Regression, and weight emerged as the most
influential factor — offering clear, actionable insight for automotive
design and consumer decision-making.

---
*Submitted as part of the IBM SkillsBuild Data Analytics with AI Academic
Internship Program, conducted by BharatCares in association with AICTE.*
